## Extract first 20 imgs from tiff stack

In [1]:
import tifffile as tiff
import numpy as np
import pandas as pd

In [2]:
def remove_outlier_frames(label_stack, thr_multiplier = 20):
    bin_imgs = np.copy(label_stack)
    bin_imgs[label_stack != 0] = 1

    min_differences = []
    for i, img in enumerate(bin_imgs):
        if i == 0:
            diff = np.abs(img - bin_imgs[i + 1])
            sad = np.sum(diff)
            min_differences.append(sad)
        elif i == len(bin_imgs)-1:
            diff = np.abs(bin_imgs[i - 1] - img)
            sad = np.sum(diff)
            min_differences.append(sad)
        else:
            diff1 = np.abs(img - bin_imgs[i + 1])
            diff2 = np.abs(img - bin_imgs[i - 1])
            sad1 = np.sum(diff1)
            sad2 = np.sum(diff2)
    
            min_sad = min(sad1, sad2)
            min_differences.append(min_sad)
    # Convert to numpy arrays
    min_differences = np.array(min_differences)

    median = np.median(min_differences)
    mad = np.median(np.abs(min_differences - median))
    threshold = median + thr_multiplier * mad

    # Find outlier indices
    outlier_indices = np.where(min_differences > threshold)[0]
    
    # Replace outlier frames with zero
    cleaned_stack = np.copy(label_stack)
    for idx in outlier_indices:
        cleaned_stack[idx] = np.zeros_like(label_stack[idx])
    
    return cleaned_stack, outlier_indices

In [3]:
IMG_PATH = '/mnt/imaging.data/PertzLab/apoDetection/TIFFs/Exp01_Site01.tif'
MASK_PATH = '/home/nbahou/myimaging/apoDet/data/dataset1/apo_masks/Exp01_Site01.npz'




img_stack = tiff.imread(IMG_PATH)
imgs = img_stack[:100]
tiff.imwrite('/home/nbahou/myimaging/apoDet/data/01_01_short_imgs_100.tif', imgs)


img_stack = np.load(MASK_PATH)['gt']
imgs = img_stack[:100]
# imgs, indeces = remove_outlier_frames(imgs)

# Experiment: replace bad frames with zeros
#imgs[40,:,:] = 0
#imgs[70,:,:] = 0
tiff.imwrite('/home/nbahou/myimaging/apoDet/data/01_01_short_mask_100.tif', imgs)

#img_stack = tiff.imread(IMG_PATH2)
#imgs = img_stack[:20]
#tiff.imwrite('/home/nbahou/myimaging/apoDet/data/06_01_short.tif', imgs)



In [7]:
print(indeces)

[]


In [5]:
# create binary mask
bin_imgs = np.copy(imgs)
bin_imgs[imgs != 0] = 1

last = bin_imgs[39]
current = bin_imgs[40]
next_img = bin_imgs[41]

diff1 = np.abs(last - current)
diff2 = np.abs(current - next_img)
ref_diff = np.abs(last - next_img)

sad_1 = np.sum(diff1) / 1000000
sad_2 = np.sum(diff2) / 1000000
sad_3 = np.sum(ref_diff) / 1000000

print(f'Sum absolute distances frame 40: {sad_1} and {sad_2}')
print(f'Sum absolute distances reference: {sad_3}')




Sum absolute distances frame 40: 16164.514249 and 14887.099921
Sum absolute distances reference: 3279.620058


In [16]:
IMG_PATH1 = '/home/nbahou/myimaging/apoDet/data/dataset1/apo_masks/Exp07_Site01.npz'
IMG_PATH2 = '/home/nbahou/myimaging/apoDet/data/dataset1/apo_masks/Exp06_Site01.npz'

path_list = [IMG_PATH1, IMG_PATH2]


img_stack = np.load(IMG_PATH1)['gt']
imgs = img_stack[:100]
tiff.imwrite('/home/nbahou/myimaging/apoDet/data/07_01_short_mask.tif', imgs)

#img_stack = np.load(IMG_PATH2)['gt']
#imgs = img_stack[:20]
#tiff.imwrite('/home/nbahou/myimaging/apoDet/data/06_01_short.tif', imgs)



In [9]:
from scipy.spatial import cKDTree

# Load your DataFrame (example data)
df = pd.read_csv('/home/nbahou/myimaging/apoDet/data/dataset1/summary_dfs/Exp01_Site01_pd_df.csv')

# Calculate average nearest-neighbor distance per time group
average_distances = []

for time, group in df.groupby('t'):
    points = group[['x', 'y']].values
    tree = cKDTree(points)
    
    # Query nearest neighbor (k=2: the point itself + nearest neighbor)
    distances, _ = tree.query(points, k=2)
    nn_distances = distances[:, 1]  # Exclude self-distance (k=0)
    
    avg_distance = np.mean(nn_distances)
    average_distances.append(avg_distance)

# Overall average across all time points
overall_average = np.mean(average_distances)
print(f"Average nearest-neighbor distance: {overall_average:.4f}")

Average nearest-neighbor distance: 37.8091


In [10]:
# Load your DataFrame (example data)
df = pd.read_csv('/home/nbahou/myimaging/apoDet/data/dataset1/summary_dfs/Exp06_Site01_pd_df.csv')

# Calculate average nearest-neighbor distance per time group
average_distances = []

for time, group in df.groupby('t'):
    points = group[['x', 'y']].values
    tree = cKDTree(points)
    
    # Query nearest neighbor (k=2: the point itself + nearest neighbor)
    distances, _ = tree.query(points, k=2)
    nn_distances = distances[:, 1]  # Exclude self-distance (k=0)
    
    avg_distance = np.mean(nn_distances)
    average_distances.append(avg_distance)

# Overall average across all time points
overall_average = np.mean(average_distances)
print(f"Average nearest-neighbor distance: {overall_average:.4f}")

Average nearest-neighbor distance: 21.6212


In [11]:
# Load your DataFrame (example data)
df = pd.read_csv('/home/nbahou/myimaging/apoDet/data/dataset1/summary_dfs/Exp07_Site01_pd_df.csv')

# Calculate average nearest-neighbor distance per time group
average_distances = []

for time, group in df.groupby('t'):
    points = group[['x', 'y']].values
    tree = cKDTree(points)
    
    # Query nearest neighbor (k=2: the point itself + nearest neighbor)
    distances, _ = tree.query(points, k=2)
    nn_distances = distances[:, 1]  # Exclude self-distance (k=0)
    
    avg_distance = np.mean(nn_distances)
    average_distances.append(avg_distance)

# Overall average across all time points
overall_average = np.mean(average_distances)
print(f"Average nearest-neighbor distance: {overall_average:.4f}")

Average nearest-neighbor distance: 18.8531


In [12]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

# Load your DataFrame
df = pd.read_csv('/home/nbahou/myimaging/apoDet/data/dataset1/summary_dfs/Exp06_Site01_pd_df.csv')

# Calculate average nearest-neighbor distance per time group
time_distances = {}
all_distances = []

for time, group in df.groupby('t'):
    if len(group) < 2:  # Skip groups with fewer than 2 points
        continue
        
    points = group[['x', 'y']].values
    tree = cKDTree(points)
    
    # Query nearest neighbor (k=2: the point itself + nearest neighbor)
    distances, _ = tree.query(points, k=2)
    nn_distances = distances[:, 1]  # Exclude self-distance at index 0
    
    time_distances[time] = np.mean(nn_distances)
    all_distances.extend(nn_distances)

# Overall average across all points (not just averaging the averages)
overall_average = np.mean(all_distances)
print(f"Average nearest-neighbor distance across all points: {overall_average:.4f}")

# Also print average per time point for comparison
for time, avg_dist in sorted(time_distances.items()):
    print(f"Time {time}: Average distance = {avg_dist:.4f}")

Average nearest-neighbor distance across all points: 21.6208
Time 0: Average distance = 21.6527
Time 1: Average distance = 21.7109
Time 2: Average distance = 21.6571
Time 3: Average distance = 21.7126
Time 4: Average distance = 21.6973
Time 5: Average distance = 21.6655
Time 6: Average distance = 21.7191
Time 7: Average distance = 21.7810
Time 8: Average distance = 21.7327
Time 9: Average distance = 21.6668
Time 10: Average distance = 21.6173
Time 11: Average distance = 21.6754
Time 12: Average distance = 21.6846
Time 13: Average distance = 21.6079
Time 14: Average distance = 21.6513
Time 15: Average distance = 21.5856
Time 16: Average distance = 21.6364
Time 17: Average distance = 21.6568
Time 18: Average distance = 21.6279
Time 19: Average distance = 21.6010
Time 20: Average distance = 21.6101
Time 21: Average distance = 21.5687
Time 22: Average distance = 21.6136
Time 23: Average distance = 21.6408
Time 24: Average distance = 21.6077
Time 25: Average distance = 21.6927
Time 26: Aver